In [13]:
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver  #导入对应的类


qwen = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen3.6-plus",
    temperature=0.7,
)

# 定义ThreadId
configId = {"configurable":{"thread_id":"1"}}

agent = create_agent(
    model=qwen,
    # 指定checkPoint
    checkpointer=InMemorySaver(),
)

response = agent.invoke(
    input={"messages": [HumanMessage("你好,我叫tim")]},
    config=configId
)

for message in response['messages']:
    message.pretty_print()


response = agent.invoke(
    input={"messages": [HumanMessage("你好,我叫什么")]},
    config=configId
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你好,我叫tim
================================== Ai Message ==================================

你好，Tim！很高兴认识你。有什么我可以帮你的吗？或者你想聊点什么？
================================ Human Message =================================

你好,我叫tim
================================== Ai Message ==================================

你好，Tim！很高兴认识你。有什么我可以帮你的吗？或者你想聊点什么？
================================ Human Message =================================

你好,我叫什么
================================== Ai Message ==================================

你叫 **Tim** 呀！有什么我可以帮你的吗？


In [16]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents.middleware import SummarizationMiddleware

# 连接sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()


# 定义ThreadId
configId = {"configurable": {"thread_id": "3"}}

summarizationMiddleware = SummarizationMiddleware(
    model=qwen,  #指定模型
    trigger=("messages",3), #触发器, 当消息达到3条时触发,这里是人消息数+AI回复消息数
    keep=("messages",1), #压缩后保留的消息数
)

agent = create_agent(
    model=qwen,
    # 指定checkPoint
    checkpointer=checkpointer,
    middleware=[summarizationMiddleware],

)

response = agent.invoke(input={"messages": [HumanMessage("你好,我叫tim")]},config=configId)
response = agent.invoke(input={"messages": [HumanMessage("我喜欢编程")]},config=configId)
response = agent.invoke(input={"messages": [HumanMessage("我是中国人")]},config=configId)


response = agent.invoke(
    input={"messages": [HumanMessage("我叫什么")]},
    config=configId
)

for message in response['messages']:
    message.pretty_print()




================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
Provide customized, actionable programming guidance and support to user Tim. All communication must be conducted strictly in Chinese.

## SUMMARY
- User confirmed nationality/language preference ("我是中国人"). AI acknowledged and committed to conducting the entire session in Chinese.
- Session remains in the initial information-gathering phase. AI is still awaiting three critical details to proceed: 1) Preferred programming languages/tech stack, 2) Current experience level/stage, 3) Specific goals or current pain points.
- No technical decisions, architectures, code, or strategies have been established. The workflow is paused pending the user's detailed response.

## ARTIFACTS
None

## NEXT STEPS
Await Tim's response detailing his tech stack, experience level, and specific programming objectives. Once provided, immediately deliver customized ac